# ESM-DMS real-data analysis

This notebook runs the real-data ESM-DMS workflow for the cellular datasets `TpoR`, `Ube4b`, and `BRCA1`, and the viral datasets `BF520` and `BG505`.

The workflow is:

1. Register the real raw-data inputs for each dataset.
2. Process raw DMS data into sequence/frequency tables.
3. Use the cached ESM sequence embeddings in `data/sequence_data/{dataset}`.
4. Load cached inference results, or recompute them from the compact embedding files.
5. Summarize replicate consistency and write figures/tables under this directory.


In [ ]:
from pathlib import Path
import itertools
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    # Allows the notebook to be run from data/esm_data_analysis/ as well as repo root.
    REPO_ROOT = Path.cwd().parents[1]

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
SEQUENCE_DIR = DATA_DIR / "sequence_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
print(REPO_ROOT)


In [ ]:
from esmDMS import CellularDMSInput, ViralDMSInput, ESMDMSConfig, esmDMS
from esmdmsfunctions import load_inference_df
from mega_analysis import _detect_layers, run_inference
from analysis_helpers import plot_cross_replicate_consistency


## Dataset registry

Cellular datasets use MaveDB nucleotide-count files. Viral datasets use paired mutant DNA and mutant virus codon-count files.

In [ ]:
CELLULAR_DATASETS = {
    "TpoR": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
    "Ube4b": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "Ube4b_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "Ube4b_nucleotide_counts.csv",
    ),
    "BRCA1": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "BRCA1_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "BRCA1_nucleotide_counts.csv",
    ),
}

VIRAL_DATASETS = {
    "BF520": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BF520_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BF520_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
    "BG505": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BG505_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BG505_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
}

DATASETS = {**CELLULAR_DATASETS, **VIRAL_DATASETS}
DATASET_KIND = {
    **{name: "cellular" for name in CELLULAR_DATASETS},
    **{name: "viral" for name in VIRAL_DATASETS},
}

pd.DataFrame(
    {"dataset": name, "kind": DATASET_KIND[name], "embedding_path": str(SEQUENCE_DIR / name)}
    for name in DATASETS
)


## Controls

`LAYERS = None` means use every cached layer found for each dataset. Set `LAYERS` to a shorter list while iterating. `FORCE_RECOMPUTE = False` reuses `inference_results*.pkl` when available.

In [ ]:
LAYERS = None
NORMALIZE = "by_layer_dim"  # one of: "none", "by_layer", "by_layer_dim"
REPLICATES = None
FORCE_RECOMPUTE = False
EVERY_N_PLOT = 6

BASE_CONFIG = ESMDMSConfig(
    embedding_model="esm2_t33_650M_UR50D",
    embedding_method="mean_pool",
    local_or_disk="disk",
    save_dir=str(SEQUENCE_DIR),
)

inference_cfg = {
    "layers": LAYERS,
    "normalize": NORMALIZE,
    "replicates": REPLICATES,
    "every_n": EVERY_N_PLOT,
}


## Process raw data

This confirms that every listed dataset can be parsed from the real raw files. The existing cached embeddings and compact inference files are used in later sections.

In [ ]:
processed = {}
processing_rows = []

for dataset, input_data in DATASETS.items():
    cfg = ESMDMSConfig(
        embedding_model=BASE_CONFIG.embedding_model,
        embedding_method=BASE_CONFIG.embedding_method,
        local_or_disk=BASE_CONFIG.local_or_disk,
        save_dir=str(SEQUENCE_DIR / dataset),
        dataset_name=dataset,
    )
    runner = esmDMS(input_data=input_data, config=cfg)
    runner.process_raw_data(drop_stop_codons=True)
    processed[dataset] = runner

    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "raw_processing_summary.csv", index=False)
processing_summary


## Validate cached embeddings

In [ ]:
cache_rows = []

for dataset in DATASETS:
    path = SEQUENCE_DIR / dataset
    detected_layers = _detect_layers(str(path))
    raw_embedding_file = path / f"{dataset}_embeddings.pkl"
    metadata_file = path / "inference_metadata.pkl"
    seq_map_file = path / "seq_id_map.pkl"

    cache_rows.append({
        "dataset": dataset,
        "embedding_path": str(path),
        "raw_embeddings": raw_embedding_file.exists(),
        "metadata": metadata_file.exists(),
        "seq_id_map": seq_map_file.exists(),
        "n_layers_detected": len(detected_layers),
        "first_layer": min(detected_layers) if detected_layers else np.nan,
        "last_layer": max(detected_layers) if detected_layers else np.nan,
    })

cache_summary = pd.DataFrame(cache_rows)
cache_summary.to_csv(TABLE_DIR / "embedding_cache_summary.csv", index=False)
cache_summary


## Run or load ESM-DMS inference

By default this loads cached `inference_results_by_layer_dim.pkl` files when they exist. If a cache is absent or stale, `run_inference` rebuilds that dataset from the compact `layer{i}_seq_to_emb.pkl` and `inference_metadata.pkl` files.

In [ ]:
paths = {dataset: str(SEQUENCE_DIR / dataset) for dataset in DATASETS}
all_results = {}

for dataset, embedding_path in paths.items():
    cfg = dict(inference_cfg)
    if cfg["layers"] is None:
        cfg["layers"] = _detect_layers(embedding_path)
    print(f"{dataset}: {len(cfg['layers'])} layers from {embedding_path}")
    all_results[dataset] = run_inference(
        embedding_path,
        cfg,
        save_results=True,
        force_recompute=FORCE_RECOMPUTE,
    )


## Inference result summary

In [ ]:
inference_rows = []

for dataset, results in all_results.items():
    detailed = results[2]
    for layer, layer_result in sorted(detailed.items()):
        s, s_joint, error_bars, s_joint_error_bars, icov, gamma_opt = layer_result
        inference_rows.append({
            "dataset": dataset,
            "kind": DATASET_KIND[dataset],
            "layer": layer,
            "n_replicates": s.shape[0],
            "n_dimensions": s.shape[1],
            "gamma_opt": gamma_opt,
            "s_joint_mean": float(np.nanmean(s_joint)),
            "s_joint_std": float(np.nanstd(s_joint)),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "inference_result_summary.csv", index=False)
inference_summary.head()


## Replicate consistency summaries

In [ ]:
def safe_corr(x, y, corr_fn):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return corr_fn(x, y).statistic


def feature_matrix_for_layer(dataset, layer, normalize=NORMALIZE):
    with open(SEQUENCE_DIR / dataset / f"layer{layer}_seq_to_emb.pkl", "rb") as handle:
        features = pickle.load(handle)
    features = np.asarray(features)
    if normalize == "by_layer":
        mean = np.mean(features)
        std = np.std(features)
        features = (features - mean) / (std if std > 0 else 1.0)
    elif normalize == "by_layer_dim":
        means = np.mean(features, axis=0)
        stds = np.std(features, axis=0)
        stds[stds == 0] = 1.0
        features = (features - means) / stds
    return features


consistency_rows = []

for dataset, results in all_results.items():
    for layer, layer_result in sorted(results[2].items()):
        s = layer_result[0]
        features = feature_matrix_for_layer(dataset, layer)
        rep_fitness = np.asarray([features @ s_rep for s_rep in s])
        for i, j in itertools.combinations(range(s.shape[0]), 2):
            consistency_rows.append({
                "dataset": dataset,
                "kind": DATASET_KIND[dataset],
                "layer": layer,
                "replicate_i": i + 1,
                "replicate_j": j + 1,
                "selection_pearson": safe_corr(s[i], s[j], pearsonr),
                "selection_spearman": safe_corr(s[i], s[j], spearmanr),
                "fitness_pearson": safe_corr(rep_fitness[i], rep_fitness[j], pearsonr),
                "fitness_spearman": safe_corr(rep_fitness[i], rep_fitness[j], spearmanr),
            })

consistency = pd.DataFrame(consistency_rows)
consistency.to_csv(TABLE_DIR / "replicate_pair_consistency.csv", index=False)

consistency_summary = (
    consistency
    .groupby(["dataset", "kind", "layer"], as_index=False)
    [["selection_pearson", "selection_spearman", "fitness_pearson", "fitness_spearman"]]
    .mean()
)
consistency_summary.to_csv(TABLE_DIR / "replicate_consistency_summary.csv", index=False)
consistency_summary.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

sns.lineplot(
    data=consistency_summary,
    x="layer",
    y="selection_pearson",
    hue="dataset",
    style="kind",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Selection coefficient replicate consistency")
axes[0].set_ylabel("Mean pairwise Pearson r")

sns.lineplot(
    data=consistency_summary,
    x="layer",
    y="fitness_pearson",
    hue="dataset",
    style="kind",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Inferred fitness replicate consistency")
axes[1].set_ylabel("Mean pairwise Pearson r")

for ax in axes:
    ax.set_ylim(-0.1, 1.05)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)

fig.tight_layout()
fig.savefig(FIGURE_DIR / "replicate_consistency_by_layer.png", dpi=150, bbox_inches="tight")
plt.show()


## Per-dataset cross-replicate plots

This uses the shared plotting helper and writes heatmaps, scatter grids, and per-layer summaries under `data/esm_data_analysis/figures/cross_replicate`.

In [ ]:
cross_rep_dir = FIGURE_DIR / "cross_replicate"
all_layers = sorted(set().union(*(result[2].keys() for result in all_results.values())))

plot_cross_replicate_consistency(
    all_results,
    all_layers,
    output_dir=str(cross_rep_dir),
    normalize=NORMALIZE,
    every_n=EVERY_N_PLOT,
)


## Optional shuffled-frequency control

Set `RUN_SHUFFLED_CONTROL = True` to break the embedding/frequency association within each replicate and generation, rerun inference, and compare cross-replicate consistency against the real data. This is computationally heavier than loading cached real-data inference.

In [ ]:
RUN_SHUFFLED_CONTROL = False

if RUN_SHUFFLED_CONTROL:
    from mega_analysis import plot_shuffled_frequencies_analysis

    plot_shuffled_frequencies_analysis(
        all_results,
        paths,
        inference_cfg,
        output_dir=str(FIGURE_DIR / "shuffled_frequency_control"),
    )
